In [5]:
import pandas as pd
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import AutoModelForSequenceClassification, AutoConfig, AutoTokenizer
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import Dataset
from torch import nn
from tqdm import tqdm
import sys

In [6]:
# load the data
with open("../../01_data/annotations_reduced.json", "r") as f:
    data = json.load(f)

# initialize dictionary for the sentiment classes
sent_dict = set()

# loop through all sentences
for task in data:
    if task["annotations"]:
        for annotation in task["annotations"]:
            label = annotation["tag"][3:]
            sent_dict.add(label)

# sort the tag dictionary
label_list = sorted(sent_dict)

# dictionaries that convert from id to tag and vice versa
label_to_id = {tag: i for i, tag in enumerate(label_list)}
id_to_label = {id: label for label, id in label_to_id.items()}

In [11]:
# get all data with annotations
data_with_annotations = []
for task in data:
    if task["annotations"]:
        data_with_annotations.append(task)

# split into training and test dataset
train_dataset, test_dataset = train_test_split(data_with_annotations, test_size=0.2, random_state=42)

class StanceDataset(Dataset):
    def __init__(self, data, tokenizer, label2id, max_len=128):
        self.dataset = []
        for item in data:
            sentence = item["sentence"]
            for ann in item["annotations"]:
                span_text = ann["text"]
                label = label2id[ann["tag"][3:]]
                # combine sentence and target span
                encoded = tokenizer(
                    sentence,
                    span_text,
                    truncation=True,
                    padding="max_length",
                    max_length=max_len,
                    return_tensors="pt"
                )
                self.dataset.append({
                    "input_ids": encoded["input_ids"].squeeze(0),
                    "attention_mask": encoded["attention_mask"].squeeze(0),
                    "label": torch.tensor(label, dtype=torch.long)
                })
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        return self.dataset[idx]

In [12]:
num_labels = len(label_to_id)

# use mps if available, otherwise cpu
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model_names = ["roberta-base", "bert-base-cased", "distilbert-base-cased"]
test_metrics = {}

for model_name in model_names:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_dataset_tensor = StanceDataset(train_dataset, tokenizer, label_to_id)
    test_dataset_tensor = StanceDataset(test_dataset, tokenizer, label_to_id)

    train_loader = DataLoader(train_dataset_tensor, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_dataset_tensor, batch_size=16, shuffle=False)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels).to(device)

    optimizer = AdamW(model.parameters(), lr=2e-5)
    criterion = torch.nn.CrossEntropyLoss()

    num_epochs = 10

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        
        for batch in progress:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            progress.set_postfix(loss=loss.item())

        print(f"Average loss: {total_loss / len(train_loader):.4f}")
    
    model.eval()
    true_labels, pred_labels = [], []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)

            true_labels.extend(labels.cpu().tolist())
            pred_labels.extend(preds.cpu().tolist())

    metrics = classification_report(
        [list(label_to_id.keys())[i] for i in true_labels],
        [list(label_to_id.keys())[i] for i in pred_labels],
        output_dict=True
        )
    test_metrics[model_name] = {
        "negative": metrics["neg"]["f1-score"],
        "neutral": metrics["neutral"]["f1-score"],
        "positive": metrics["pos"]["f1-score"]
    }

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 1/10: 100%|██████████| 103/103 [00:39<00:00,  2.59it/s, loss=0.717]


Average loss: 0.7799


Epoch 2/10: 100%|██████████| 103/103 [00:40<00:00,  2.52it/s, loss=0.606]


Average loss: 0.5663


Epoch 3/10: 100%|██████████| 103/103 [00:40<00:00,  2.54it/s, loss=0.463]


...

Average loss: 0.0498


Epoch 9/10: 100%|██████████| 103/103 [00:19<00:00,  5.27it/s, loss=0.00506]


Average loss: 0.0467


Epoch 10/10: 100%|██████████| 103/103 [00:19<00:00,  5.26it/s, loss=0.0035] 


Average loss: 0.0330


In [14]:
with open("evaluation_metrics_bert.json", "w") as f:
    json.dump(test_metrics, f, indent=4)